# Google QUEST: full-data DeBERTa training

Train a shared DeBERTa-v3-base encoder on title/body and title/answer pairs, using a question-group holdout and all 30 quality targets. The run saves its best checkpoint, tokenizer, held-out predictions and test predictions with input/code fingerprints.

This private training notebook uses internet access to obtain the pinned public pretrained model. Competition scoring uses the separate offline inference notebook and an attached trained-run asset. Local holdout Spearman and leaderboard Spearman measure different populations.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/quest-huggingface"
wheels = list(Path("/kaggle/input").rglob("quest_labeling-*.whl"))
assert len(wheels) == 1, "Attach exactly one QUEST code wheel"
assets = wheels[0].parent
source = json.loads((assets / "source.json").read_text())
assert hashlib.sha256(wheels[0].read_bytes()).hexdigest() == source["wheel_sha256"]
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                "--disable-pip-version-check", str(wheels[0])], check=True)
print({"python": sys.version.split()[0], "versions": {name: importlib.metadata.version(name)
       for name in ["torch", "transformers", "numpy", "pandas", "scikit-learn"]},
       "py7zr": importlib.metadata.version("py7zr") if importlib.util.find_spec("py7zr") else None,
       "code": source})


In [ ]:
import torch
from quest.runtime import train
assert torch.cuda.is_available(), "Run this full recipe with a GPU"
competition_roots = [p.parent for p in Path("/kaggle/input").rglob("train.csv")
                     if (p.parent / "test.csv").is_file() and
                     (p.parent / "sample_submission.csv").is_file()]
assert len(competition_roots) == 1, "Attach the official Google QUEST competition"
config = json.loads((assets / "deberta.json").read_text())
if "fold_plan" in config:
    config["fold_plan"]["path"] = str(assets / config["fold_plan"]["path"])
manifest = train(competition_roots[0], Path("/kaggle/working/quest-run"), config)
print(json.dumps({k: manifest[k] for k in ["status", "train_rows", "validation_rows",
                  "test_rows", "best_validation_spearman", "elapsed_seconds"]}, indent=2))
